# Glomeruli Segmentation — Training on Colab

### Before running
1. Zip the `data/dataset/` folder from your local machine.
   The zip must contain `train/`, `validation/`, `test/` at the root level:
   ```
   dataset.zip
   ├── train/
   │   ├── img/
   │   └── mask/
   ├── validation/
   │   ├── img/
   │   └── mask/
   └── test/
       ├── img/
       └── mask/
   ```
2. Upload `dataset.zip` to Google Drive (e.g. `My Drive/Glomeruli/dataset.zip`).
3. In Colab: **Runtime → Change runtime type → T4 GPU**.
4. Run all cells in order.

In [ ]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration
Edit the paths and hyperparameters below before continuing.

In [ ]:
# --- Paths (edit these) ---
DATASET_ZIP  = '/content/drive/MyDrive/Glomeruli/dataset.zip'
OUTPUT_DIR   = '/content/drive/MyDrive/Glomeruli/models'

# --- Hyperparameters ---
EPOCHS     = 10
BATCH_SIZE = 8
LR         = 0.1

## Environment setup
Clone the repo and extract the dataset.

In [ ]:
import os, sys, zipfile

REPO_DIR    = '/content/Glomeruli-FP03-2026'
DATASET_DIR = '/content/dataset'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/PasqualeSarcina/Glomeruli-FP03-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q scikit-image tqdm

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Extracting dataset...')
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall(DATASET_DIR)
print('Done.')

# Verify structure
for split in ('train', 'validation', 'test'):
    n = len(list(__import__('pathlib').Path(f'{DATASET_DIR}/{split}/img').glob('*.png')))
    print(f'  {split}: {n} images')

## Training

In [ ]:
import keras
from pathlib import Path

from src.segmentation.dataset import SegmentationDataset
from src.segmentation.segnet import build_segnet_vgg19, compile_segnet, lr_step_decay

train_ds = SegmentationDataset(
    Path(DATASET_DIR) / 'train',
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
).build()

val_ds = SegmentationDataset(
    Path(DATASET_DIR) / 'validation',
    batch_size=BATCH_SIZE,
    shuffle=False,
    augment=False,
).build()

model = build_segnet_vgg19()
model = compile_segnet(model, initial_lr=LR)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        f'{OUTPUT_DIR}/best_model.keras',
        save_best_only=True,
        monitor='val_mean_io_u',
        mode='max',
        verbose=1,
    ),
    keras.callbacks.LearningRateScheduler(lr_step_decay, verbose=1),
    keras.callbacks.CSVLogger(f'{OUTPUT_DIR}/training_log.csv'),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(f'{OUTPUT_DIR}/final_model.keras')
print('Model saved.')

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history.history['mean_io_u'], label='train')
axes[2].plot(history.history['val_mean_io_u'], label='val')
axes[2].set_title('Mean IoU')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()